# 📊 Proyecto EDA – Análisis del comportamiento de pago de créditos

In [1]:
# ==========================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ==========================================

import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
sns.set_theme(style="whitegrid")

# Mostrar todas las columnas
pd.set_option("display.max_columns", None)

# Mostrar números con 2 decimales
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 📁 2. Carga de datos


In [3]:
# ==========================================
# 2. CARGA DE DATOS
# ==========================================

ruta_archivo = r"C:\Proyectos_soy_henry\M5\proyecto_integrador_M5\PIM5_V1\Base_de_datos.xlsx"

df = pd.read_excel(ruta_archivo)

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

Filas: 10,763
Columnas: 23


## 🔍 3. Revision Inicial

In [4]:
df.head()

,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,puntaje_datacredito,cant_creditosvigentes,huella_consulta,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,"3,692,160.00",10,42,Independiente,8000000,2500000,341296,88.77,695.00,10,5,0.00,"51,258.00","51,258.00",0.00,5,0,0,"908,526.00",Estable,1
1,4,2025-04-22 09:47:35,"840,000.00",6,60,Empleado,3000000,2000000,124876,95.23,789.00,3,1,0.00,"8,673.00","8,673.00",0.00,0,0,2,"939,017.00",Creciente,1
2,9,2026-01-08 12:22:40,"5,974,028.40",10,36,Independiente,4036000,829000,529554,47.61,740.00,4,5,0.00,"18,702.00","18,702.00",0.00,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,"1,671,240.00",6,48,Empleado,1524547,498000,252420,95.23,837.00,4,4,0.00,"15,782.00","15,782.00",0.00,3,0,0,"1,536,193.00",Creciente,1
4,9,2025-04-26 11:24:26,"2,781,636.00",11,44,Empleado,5000000,4000000,217037,95.23,771.00,4,6,0.00,"204,804.00","204,804.00",0.00,3,0,1,"933,473.00",Creciente,1


In [ ]:
df.tail()

In [ ]:
df.sample(10, random_state=42)

## 🧬 4. Comprensión del esquema de datos

In [ ]:
df.info()

In [ ]:
diccionario_datos = pd.DataFrame({
    "Variable": df.columns,
    "Tipo": df.dtypes.astype(str),
    "Valores_nulos": df.isnull().sum().values,
    "Porcentaje_nulos": (df.isnull().mean() * 100).round(2).values,
    "Valores_unicos": df.nunique().values
})

diccionario_datos

### Resumen General del Dataset
Volumen de datos: El conjunto contiene aproximadamente 10,764 registros (calculado a partir de los recuentos y porcentajes de nulos) distribuidos en 23 variables.

Calidad global: La calidad de la información es muy alta. 17 de las 23 variables no presentan valores nulos (100% de registros completos).

### Análisis de Valores Nulos (Faltantes)
Podemos agrupar las variables según su porcentaje de nulos:

Sin nulos (0.00%): Información clave del crédito (tipo_credito, fecha_prestamo, capital_prestado, plazo_meses, cuota_pactada), perfil del cliente (edad_cliente, salario_cliente, tipo_laboral) y variables del sector financiero.

Nulos marginales (< 6%):

puntaje_datacredito: 6 nulos (0.06%).

saldo_mora y saldo_total: 156 nulos (1.45%).

saldo_principal: 405 nulos (3.76%).

saldo_mora_codeudor: 590 nulos (5.48%).

Nulos significativos (~27.2%):

promedio_ingresos_datacredito (2,930 nulos / 27.22%) y tendencia_ingresos (2,932 nulos / 27.24%).

Interpretación de negocio: Es habitual que cerca del 27% de los clientes no cuenten con información estimada de ingresos en la central de riesgo (ej. por ser informales, no bancarizados o carecer de historial suficiente).

### 📋 Resultado de Imputaciones
No se realizaran imputaciones toda vez que las variables con nulos no se tendran en cuenta en el modelo

In [ ]:
plt.figure(figsize=(12, 6))

nulos_pct = df.isnull().mean().sort_values(ascending=False) * 100

nulos_pct[nulos_pct > 0].plot(kind="bar")

plt.title("Porcentaje de valores faltantes por variable")
plt.ylabel("Porcentaje (%)")
plt.xlabel("Variable")

plt.xticks(rotation=45)
plt.show()

## ⚙️ 6. Ajuste de tipos de datos

In [ ]:
df.dtypes

In [ ]:
#Ajustamos la columna fecha_prestamo a tipo datetime para poder trabajar con ella de manera más eficiente.
df["fecha_prestamo"] = pd.to_datetime(
    df["fecha_prestamo"],
    errors="coerce"
)

In [ ]:
# Variables categoricas
columnas_categoricas = [
    "tipo_credito",
    "tipo_laboral",
    "tendencia_ingresos"
]

for col in columnas_categoricas:
    df[col] = df[col].astype("category")

df.dtypes

## 🔬 7. Análisis de la variable objetivo

In [ ]:
# Variable objetivo
df["Pago_atiempo"].value_counts()

In [ ]:
df["Pago_atiempo"].value_counts(normalize=True) * 100

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x="Pago_atiempo"
)

plt.title("Distribución del pago oportuno")
plt.xlabel("Pago a tiempo")
plt.ylabel("Cantidad de clientes")

plt.show()

### Se evidencia desbalanceo de clases
Pago_atiempo
1   95.25
0    4.75

## 📌 8.Análisis univariable

In [ ]:
#Variables numéricas
variables_numericas = df.select_dtypes(
    include=["int64", "float64"]
).columns

variables_numericas

In [ ]:
# Estadisticas descriptivas
df[variables_numericas].describe().T

In [ ]:
estadisticas = pd.DataFrame(index=variables_numericas)

estadisticas["media"] = df[variables_numericas].mean()
estadisticas["mediana"] = df[variables_numericas].median()
estadisticas["moda"] = df[variables_numericas].mode().iloc[0]
estadisticas["min"] = df[variables_numericas].min()
estadisticas["max"] = df[variables_numericas].max()
estadisticas["rango"] = (
    df[variables_numericas].max()
    - df[variables_numericas].min()
)
estadisticas["desv_std"] = df[variables_numericas].std()
estadisticas["varianza"] = df[variables_numericas].var()
estadisticas["Q1"] = df[variables_numericas].quantile(0.25)
estadisticas["Q3"] = df[variables_numericas].quantile(0.75)

estadisticas["IQR"] = (
    estadisticas["Q3"]
    - estadisticas["Q1"]
)

estadisticas["skewness"] = df[variables_numericas].skew()
estadisticas["kurtosis"] = df[variables_numericas].kurtosis()

estadisticas

## 📊  9. Histogramas

In [ ]:
variables_numericas_sin_target = [
    col for col in variables_numericas
    if col != "Pago_atiempo"
]

for col in variables_numericas_sin_target:

    plt.figure(figsize=(8, 5))

    sns.histplot(
        data=df,
        x=col,
        kde=True
    )

    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")

    plt.show()

## 📊 10. Boxplots

In [ ]:
for col in variables_numericas_sin_target:

    plt.figure(figsize=(8, 4))

    sns.boxplot(
        data=df,
        x=col
    )

    plt.title(f"Boxplot de {col}")

    plt.show()

##  🔍 11. Variables categóricas

In [ ]:
variables_categoricas = df.select_dtypes(
    include=["object", "category"]
).columns

variables_categoricas

In [ ]:
for col in variables_categoricas:

    print("=" * 60)
    print(f"Variable: {col}")
    print(df[col].value_counts(dropna=False))
    print()

### Graficos

In [ ]:
for col in variables_categoricas:

    plt.figure(figsize=(10, 5))

    sns.countplot(
        data=df,
        x=col,
        order=df[col].value_counts().index
    )

    plt.title(f"Distribución de {col}")
    plt.xticks(rotation=45)

    plt.show()

##  🔬 12. Análisis bivariable
    Aquí vamos a analizar las variables respecto a Pago_atiempo

In [ ]:
tabla_tipo_credito = pd.crosstab(
    df["tipo_credito"],
    df["Pago_atiempo"],
    normalize="index"
) * 100

tabla_tipo_credito

### Graficos

In [ ]:
pd.crosstab(
    df["tipo_laboral"],
    df["Pago_atiempo"],
    normalize="index"
) * 100

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=df,
    x="tipo_laboral",
    y="Pago_atiempo"
)

plt.title("Pago oportuno según tipo laboral")

plt.xticks(rotation=45)

plt.show()

## 📈  13. Variables numéricas vs objetivo
    Vamos a comparar las distribuciones de cada variable según el pago
    

In [ ]:
for col in variables_numericas_sin_target:

    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=df,
        x="Pago_atiempo",
        y=col
    )

    plt.title(
        f"{col} vs Pago oportuno"
    )

    plt.show()

## 🔍 14. Análisis multivariable
    Matriz de correlación

In [ ]:
plt.figure(figsize=(16, 12))

correlacion = df[
    variables_numericas
].corr()

sns.heatmap(
    correlacion,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Matriz de correlación")

plt.show()

## 📉  15. Pairplot
Para evitar gráficos excesivamente grandes, vamos a seleccionar las variables más relevantes

In [ ]:
variables_pairplot = [
    "capital_prestado",
    "salario_cliente",
    "cuota_pactada",
    "puntaje",
    "puntaje_datacredito",
    "saldo_mora",
    "Pago_atiempo"
]

sns.pairplot(
    df[variables_pairplot],
    hue="Pago_atiempo",
    diag_kind="hist"
)

plt.show()

## 📋  16. Tablas cruzadas multivariables

In [ ]:
tabla = pd.crosstab(
    [
        df["tipo_credito"],
        df["tipo_laboral"]
    ],
    df["Pago_atiempo"],
    normalize="index"
) * 100

tabla

## 📊 17. Análisis temporal 

In [ ]:
df["año"] = df["fecha_prestamo"].dt.year
df["mes"] = df["fecha_prestamo"].dt.month

In [ ]:
df.groupby("año")["Pago_atiempo"].mean() * 100

In [ ]:
df.groupby("mes")["Pago_atiempo"].mean() * 100

### Visualizacion

In [ ]:
plt.figure(figsize=(10, 5))

(
    df.groupby("mes")["Pago_atiempo"]
    .mean()
    .mul(100)
    .plot(kind="line", marker="o")
)

plt.title("Tasa de pago oportuno por mes")
plt.ylabel("Pago oportuno (%)")
plt.xlabel("Mes")

plt.show()

## 18. Reglas de validación de datos

edad_cliente > 0,
edad_cliente <= 100,
salario_cliente >= 0,
capital_prestado > 0,
plazo_meses > 0

##### Puntajes

In [ ]:
for col in [
    "puntaje",
    "puntaje_datacredito"
]:

    print(col)
    print(df[col].describe())

In [ ]:
reglas_validacion = {
    "edad_cliente": "Debe estar entre 18 y 100 años",
    "salario_cliente": "Debe ser mayor o igual a 0",
    "capital_prestado": "Debe ser mayor a 0",
    "plazo_meses": "Debe ser mayor a 0",
    "cuota_pactada": "Debe ser mayor a 0",
    "saldo_mora": "Debe ser mayor o igual a 0",
    "Pago_atiempo": "Debe contener únicamente 0 o 1"
}

pd.DataFrame(
    reglas_validacion.items(),
    columns=["Variable", "Regla"]
)

## 19. Variables derivadas

### Relacion cuota/ingreso

In [ ]:
df["ratio_cuota_ingreso"] = (
    df["cuota_pactada"]
    / df["salario_cliente"]
)

### Nivel de endeudamiento total

In [ ]:
df["deuda_total"] = (
    df["total_otros_prestamos"]
    + df["saldo_total"]
)

### Tiene Mora

In [ ]:
df["tiene_mora"] = (
    df["saldo_mora"] > 0
).astype(int)

### Tiene codeudor con deuda

In [ ]:
df["tiene_mora_codeudor"] = (
    df["saldo_mora_codeudor"] > 0
).astype(int)

In [ ]:
df.head()

# 20. 📊 Conclusiones del Análisis Exploratorio de Datos (EDA)

## 1. La base de datos presenta un volumen adecuado y una estructura multidimensional para el análisis predictivo

El conjunto de datos está compuesto por **10.763 registros y 23 variables**, que incluyen información relacionada con las características del crédito, perfil demográfico y laboral del cliente, ingresos, endeudamiento, comportamiento financiero e información proveniente de centrales de riesgo. La variable objetivo `Pago_atiempo` está correctamente identificada y presenta dos categorías, 0 y 1, lo que permite plantear el problema como una clasificación binaria. Esta estructura proporciona una base adecuada para desarrollar posteriormente un modelo predictivo orientado a estimar la probabilidad de pago oportuno.

## 2. La variable objetivo presenta un fuerte desbalance hacia el pago oportuno

La variable `Pago_atiempo` presenta una media de **0,9525**, lo que indica que aproximadamente el **95,25% de los registros corresponden a clientes que pagan a tiempo**, mientras que cerca del **4,75% representan casos de no pago oportuno**. Este resultado constituye uno de los hallazgos más relevantes del EDA, debido a que evidencia un **desbalance importante de clases**. Por lo tanto, en la etapa de modelamiento no será suficiente utilizar únicamente la exactitud (accuracy) como métrica de evaluación, ya que un modelo podría obtener una precisión aparentemente alta simplemente por predecir mayoritariamente la clase positiva. Será necesario considerar métricas como **precision, recall, F1-score, ROC-AUC y, especialmente, PR-AUC**, además de evaluar técnicas de balanceo cuando corresponda.

## 3. Los datos presentan una cantidad importante de valores faltantes concentrados en variables específicas

Aunque gran parte de las variables no presenta valores faltantes, existen variables que requieren un tratamiento especial. `promedio_ingresos_datacredito` tiene **2.930 valores faltantes, equivalentes al 27,22%**, y `tendencia_ingresos` presenta **2.932 valores faltantes, equivalentes al 27,24%**. Otras variables tienen niveles de ausencia menores, como `saldo_mora_codeudor` con 5,48%, `saldo_principal` con 3,76% y `saldo_mora` y `saldo_total` con 1,45%. Por lo tanto, la decisión de eliminar directamente estas variables del modelo debe ser revisada, ya que la ausencia de información podría contener información relevante sobre el perfil del cliente. Se recomienda evaluar una estrategia de imputación acompañada de una variable indicadora de ausencia (`missing indicator`), especialmente para las variables con aproximadamente 27% de datos faltantes.

## 4. Las variables financieras presentan distribuciones altamente asimétricas y presencia de valores extremos

Las variables relacionadas con el salario, otros préstamos, capital prestado, cuotas y saldos presentan una fuerte asimetría positiva. Por ejemplo, `salario_cliente` tiene una media de aproximadamente **17,2 millones**, pero una mediana de solo **3 millones**, con un máximo de **22.000 millones** y un skewness de **43,78**. De manera similar, `total_otros_prestamos` presenta una media de aproximadamente **6,24 millones**, una mediana de **1 millón** y un máximo superior a **6.787 millones**. Estos resultados evidencian la presencia de valores extremos que pueden distorsionar los análisis estadísticos y afectar algunos algoritmos de machine learning. Se recomienda investigar estos registros, determinar si corresponden a valores reales o errores de captura y, dependiendo del modelo seleccionado, evaluar transformaciones logarítmicas, winsorización o métodos robustos.

## 5. Se identifican inconsistencias que requieren reglas de validación antes del modelamiento

El análisis estadístico revela valores que deben ser revisados desde el punto de vista de calidad de datos. La variable `edad_cliente` alcanza un máximo de **123 años**, mientras que `puntaje` presenta valores mínimos negativos de aproximadamente **-38** y `puntaje_datacredito` valores mínimos de **-7**. Adicionalmente, `salario_cliente` contiene valores iguales a cero y presenta máximos extremadamente elevados. Estos resultados sugieren que no todos los valores extremos deben tratarse automáticamente como outliers estadísticos: algunos podrían representar errores de captura o inconsistencias de negocio. Por ello, antes del modelamiento se deben definir reglas de validación, por ejemplo, rangos razonables de edad, puntajes válidos y valores financieros no negativos.

## 6. La variable `tendencia_ingresos` presenta un problema de calidad que debe corregirse

El análisis de variables categóricas revela una situación especialmente importante en `tendencia_ingresos`. Aunque las categorías esperadas son `Creciente`, `Decreciente` y `Estable`, la variable también contiene valores numéricos, valores negativos y otros registros atípicos. Se identifican **5.294 registros como "Creciente"**, **1.291 como "Decreciente"**, **1.188 como "Estable"** y **2.932 valores nulos**, pero además aparecen múltiples valores numéricos que no corresponden a categorías válidas. Esto indica una posible contaminación o transformación incorrecta de la variable. Antes de utilizarla en un modelo, se debe investigar su origen, identificar qué representan esos valores numéricos y establecer una regla de limpieza o recodificación. Este hallazgo también demuestra que la validación de tipos de datos no debe limitarse a revisar si una variable es `object` o `numeric`, sino que debe comprobar la coherencia semántica de sus valores.

## 7. La variable `puntaje` presenta una distribución poco confiable que debe investigarse antes de utilizarla como predictor

El análisis descriptivo de `puntaje` muestra una mediana de **95,23**, un máximo también de **95,23**, pero un mínimo de **-38,01**, con una asimetría de **-4,87**. El hecho de que el tercer cuartil, la mediana y el máximo coincidan en 95,23, mientras existen valores negativos, sugiere que la variable podría estar limitada por un valor superior o contener una codificación especial para ciertos registros. Antes de incorporarla directamente a un modelo, es recomendable analizar su distribución detalladamente, identificar la proporción de valores negativos y determinar si estos representan valores faltantes codificados, ausencia de información o errores. Este análisis es fundamental porque una variable de score mal interpretada podría introducir ruido o sesgo en el modelo.

## 8. Las variables relacionadas con endeudamiento y comportamiento financiero tienen potencial para construir indicadores derivados de mayor valor analítico

La base contiene información sobre `total_otros_prestamos`, `cuota_pactada`, `cant_creditosvigentes`, `saldo_mora`, `saldo_total`, `saldo_principal` y créditos distribuidos por diferentes sectores. El análisis muestra una elevada dispersión y asimetría en varias de estas variables, por lo que resulta conveniente complementar los valores absolutos con indicadores relativos. Se recomienda crear variables como **ratio cuota-ingreso**, **ratio deuda-ingreso**, **ratio capital-ingreso**, **deuda total**, **número total de créditos**, **indicador de presencia de mora** y **proporción del saldo en mora sobre el saldo total**. Estas variables podrían representar mejor la capacidad de pago y el nivel de exposición financiera del cliente y deben ser evaluadas posteriormente como posibles predictores de `Pago_atiempo`.

## 9. La información laboral y crediticia permite segmentar diferentes perfiles de clientes

La población analizada está compuesta por **6.754 empleados y 4.009 independientes**, lo que representa aproximadamente un 63% y 37% respectivamente. Asimismo, existen seis códigos principales de `tipo_credito`, aunque la distribución está fuertemente concentrada en los tipos 4 y 9, que reúnen la gran mayoría de los registros. Esta concentración debe considerarse al realizar análisis bivariable y multivariable, ya que las categorías con pocos registros pueden producir resultados inestables. Se recomienda analizar `Pago_atiempo` por tipo de crédito y situación laboral, así como estudiar posibles interacciones entre estas variables y las características financieras del cliente.

## 10. El principal reto de la siguiente etapa será construir un proceso robusto de calidad, transformación y modelamiento que considere el fuerte desbalance de la variable objetivo

En conjunto, el EDA demuestra que el dataset contiene información valiosa para predecir el comportamiento de pago, pero requiere una etapa adicional de preparación antes de entrenar modelos. Las principales acciones recomendadas son: corregir las inconsistencias de `tendencia_ingresos`, validar los valores extremos y potencialmente inválidos de edad, puntajes e ingresos, analizar el tratamiento de los valores faltantes, revisar la distribución y codificación de `puntaje`, crear variables financieras derivadas y establecer un protocolo de validación de datos. Además, debido a que aproximadamente el **95,25% de los registros corresponden a pagos oportunos**, la evaluación del modelo deberá centrarse en su capacidad para identificar correctamente la clase minoritaria y no únicamente en la exactitud global. El resultado del EDA permite, por tanto, definir una hoja de ruta clara para las siguientes fases: **limpieza y validación → imputación y tratamiento de valores faltantes → feature engineering → selección de variables → balanceo de clases → modelamiento → evaluación y monitoreo**.
